In [16]:
#Test the connection
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from urllib.parse import quote_plus

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))  # URL-encode special characters

connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
print("Connecting with user:", DB_USER)  # debug check

engine = create_engine(connection_string)

query = """
SELECT id, title, price, location, constituency, address, 
       bedrooms, bathrooms, floor_size, excerpt
FROM property_listings
WHERE price IS NOT NULL
"""

df = pd.read_sql(query, engine)
print(f"Loaded {len(df)} rows from database")
df.head()

Connecting with user: ml_user_2.krdnjiphdvdfeqqrxvji
Loaded 1040 rows from database


,id,title,price,location,constituency,address,bedrooms,bathrooms,floor_size,excerpt
0,1,4 Bedroom Townhouse,30000000.0,Kiambu Road,Westlands,"9 Kiambu Rd, Kiambu Road, Nairobi",4.0,5.0,250.0,4-Bedroom All En-Suite Townhouse for Sale on K...
1,2,4 Bedroom Apartment / Flat,65000000.0,Parklands,Westlands,"PRR9+25R Parklands, Parklands, Nairobi",4.0,5.0,250.0,4-Bedroom Duplex Apartment for Rent in Parklan...
2,3,4 Bedroom Apartment / Flat,65000000.0,Parklands,Westlands,"PRR9+25R Parklands, Parklands, Nairobi",4.0,5.0,250.0,LUXURY 4-BEDROOM DUPLEX APARTMENT FOR RENT & S...
3,4,5 Bedroom Apartment / Flat,40000000.0,Kileleshwa,Langata,"Olengruone Ave Kileleshwa, Kileleshwa, Nairobi",5.0,6.0,438.0,Newly Completed 5-Bedroom Duplex Penthouse for...
4,5,5 Bedroom Apartment / Flat,52000000.0,Kilimani,Langata,"PQ4H+8MC Elgeyo Rd, Kilimani, Nairobi",5.0,7.0,359.0,"FOR SALE\nSPACIOUS 3, 4, 5 BEDROOM SIMPLEX AP..."


In [17]:
#Cell 2 -Save snapshot +inspect
import os

os.makedirs('../data', exist_ok=True)
df.to_csv('../data/property_listings_db.csv', index=False)
print(f"Saved snapshot: {len(df)} rows")

print(df.isnull().sum())
print(f"\nbedrooms range: {df['bedrooms'].min()} - {df['bedrooms'].max()}")
print(f"price range: {df['price'].min():,.0f} - {df['price'].max():,.0f}")

Saved snapshot: 1040 rows
id               0
title            0
price            0
location         0
constituency     0
address          0
bedrooms        25
bathrooms       16
floor_size      74
excerpt          0
dtype: int64

bedrooms range: 1.0 - 11.0
price range: 500,000 - 270,000,000


In [18]:
#Cell 3 — 

import re
from sklearn.preprocessing import LabelEncoder

# Dedup (safety net, should be minimal now)
df = df.drop_duplicates(subset=['title', 'price', 'address', 'floor_size'])
print(f"After dedup: {df.shape}")

# Property type from title
df['property_type'] = df['title'].str.extract(
    r'(Apartment|House|Villa|Townhouse|Studio|Penthouse)', flags=re.IGNORECASE
)
df['property_type'] = df['property_type'].fillna('Other').str.title()

print(df['property_type'].value_counts())
print(df['constituency'].value_counts())

After dedup: (620, 10)
property_type
Apartment    521
Townhouse     45
House         43
Other         11
Name: count, dtype: int64
constituency
Westlands    333
Langata      248
Mavoko        19
Unknown       14
Makadara       3
Roysambu       3
Name: count, dtype: int64


In [19]:
# Cell 4 -  One-Hot encode constituency
const_dummies = pd.get_dummies(df['constituency'], prefix='const')
df = pd.concat([df, const_dummies], axis=1)
const_cols = [c for c in df.columns if c.startswith('const_')]

# Label encode property type
le_type = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['property_type'])

feature_cols = ['bedrooms', 'bathrooms', 'floor_size', 'type_encoded'] + const_cols
print(f"Total features: {len(feature_cols)}")
print(feature_cols)

Total features: 10
['bedrooms', 'bathrooms', 'floor_size', 'type_encoded', 'const_Langata', 'const_Makadara', 'const_Mavoko', 'const_Roysambu', 'const_Unknown', 'const_Westlands']


In [20]:
# Cell 5 — Build training set + save
import pickle

target_col = 'price'
train_df = df[feature_cols + [target_col]].dropna()
print(f"Training dataset shape: {train_df.shape}")

train_df.to_csv('../data/train_ready.csv', index=False)

os.makedirs('../models', exist_ok=True)
with open('../models/encoders.pkl', 'wb') as f:
    pickle.dump({'property_type': le_type}, f)

print("✅ train_ready.csv saved")

Training dataset shape: (569, 11)
✅ train_ready.csv saved


In [21]:
#The initial data with 1700~ rows was not clean and we need to check for the outlier  then we check

# Check for outliers
print("Price distribution:")
print(train_df['price'].describe())

print("\nTop 10 highest prices:")
print(df.nlargest(10, 'price')[['title', 'price', 'floor_size', 'bedrooms', 'constituency']])

print("\nBottom 10 lowest prices:")
print(df.nsmallest(10, 'price')[['title', 'price', 'floor_size', 'bedrooms', 'constituency']])

Price distribution:
count    5.690000e+02
mean     3.146697e+07
std      4.198480e+07
min      1.060000e+06
25%      9.500000e+06
50%      1.650000e+07
75%      3.000000e+07
max      2.700000e+08
Name: price, dtype: float64

Top 10 highest prices:
                    title        price  floor_size  bedrooms constituency
738       3 Bedroom House  270000000.0       300.0       3.0      Langata
982       3 Bedroom House  270000000.0       200.0       3.0      Langata
801    Vacant Land / Plot  260000000.0         NaN       NaN      Langata
46    5 Bedroom Townhouse  220000000.0       900.0       5.0      Langata
47        5 Bedroom House  220000000.0       900.0       5.0      Langata
493   5 Bedroom Townhouse  220000000.0       900.0       5.0      Langata
570       5 Bedroom House  220000000.0       900.0       5.0      Langata
357       6 Bedroom House  195000000.0       704.0       6.0      Langata
458  11 Bedroom Townhouse  195000000.0       400.0      11.0      Langata
534       6 

In [14]:
#No need to run this cell now that DE team fixed the DB error
#Cell not required 
# Fix floor_size data entry errors (values >2000 are likely missing a decimal point) -- Run this before cell 6 
train_df = train_df[train_df['floor_size'] < 2000]
print(f"After floor_size fix: {len(train_df)} rows")

# Remove price outliers — use percentile-based filtering
lower = train_df['price'].quantile(0.02)
upper = train_df['price'].quantile(0.98)
print(f"Keeping prices between KSh {lower:,.0f} and KSh {upper:,.0f}")

train_df = train_df[(train_df['price'] >= lower) & (train_df['price'] <= upper)]
print(f"After price outlier removal: {len(train_df)} rows")

print(train_df['price'].describe())

After floor_size fix: 569 rows
Keeping prices between KSh 5,124,400 and KSh 190,000,000
After price outlier removal: 547 rows
count    5.470000e+02
mean     2.861508e+07
std      3.383296e+07
min      5.150000e+06
25%      9.700000e+06
50%      1.650000e+07
75%      2.950000e+07
max      1.900000e+08
Name: price, dtype: float64


In [22]:

# Cell 6 — Train/test split + models
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

X = train_df.drop(columns=['price'])
y = train_df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

def report(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"=== {name} ===")
    print(f"RMSE: KSh {rmse:,.0f}")
    print(f"MAE:  KSh {mae:,.0f}")
    print(f"R²:   {r2:.4f}\n")
    return rmse, mae, r2

report("Linear Regression", y_test, y_pred_lr)
report("XGBoost", y_test, y_pred_xgb)

Train: 455 rows, Test: 114 rows
=== Linear Regression ===
RMSE: KSh 18,672,359
MAE:  KSh 10,701,912
R²:   0.7664

=== XGBoost ===
RMSE: KSh 15,163,673
MAE:  KSh 6,981,156
R²:   0.8459



(np.float64(15163673.022518532), 6981156.228070175, 0.8459263727731567)

In [24]:
#Test and try the model with various values 

import pandas as pd

# Get the exact constituency columns the model was trained on
const_cols_trained = [c for c in train_df.columns if c.startswith('const_')]
print("Model trained on these constituency columns:", const_cols_trained)

# Test property
test_property = {
    'bedrooms': 3,
    'bathrooms': 2.0,
    'floor_size': 120,
    'property_type': 'Apartment',
    'constituency': 'Westlands'
}

# Encode property type
type_encoded = le_type.transform([test_property['property_type']])[0]

# Build constituency features using the EXACT columns the model expects
const_features = {col: 0 for col in const_cols_trained}
target_col = f"const_{test_property['constituency']}"
if target_col in const_features:
    const_features[target_col] = 1
else:
    print(f"Warning: {test_property['constituency']} not in training data, using baseline (no constituency flag set)")

feature_row = pd.DataFrame([{
    'bedrooms': test_property['bedrooms'],
    'bathrooms': test_property['bathrooms'],
    'floor_size': test_property['floor_size'],
    'type_encoded': type_encoded,
    **const_features
}])

# Ensure column order matches training exactly
feature_row = feature_row[X_train.columns]

predicted_price = xgb.predict(feature_row)[0]
print(f"\nPredicted price: KSh {predicted_price:,.0f}")
print(f"That's KSh {predicted_price/1e6:.2f}M")

Model trained on these constituency columns: ['const_Langata', 'const_Makadara', 'const_Mavoko', 'const_Roysambu', 'const_Unknown', 'const_Westlands']

Predicted price: KSh 15,577,827
That's KSh 15.58M


In [25]:
# Final Cell to save model
import pickle

rmse_xgb = 15163673.022518532
mae_xgb = 6981156.228070175
r2_xgb = 0.8459263727731567

with open('../models/stage1_price_model.pkl', 'wb') as f:
    pickle.dump(xgb, f)

print("✅ Model saved — trained on", len(train_df), "rows (no manual outlier filtering — DE pipeline fix handled it)")
print(f"\n=== Final XGBoost Stage 1 Metrics ===")
print(f"RMSE: KSh {rmse_xgb:,.0f}")
print(f"MAE:  KSh {mae_xgb:,.0f}")
print(f"R²:   {r2_xgb:.4f} — explains {r2_xgb*100:.1f}% of price variance")

✅ Model saved — trained on 569 rows (no manual outlier filtering — DE pipeline fix handled it)

=== Final XGBoost Stage 1 Metrics ===
RMSE: KSh 15,163,673
MAE:  KSh 6,981,156
R²:   0.8459 — explains 84.6% of price variance


In [3]:
#Checking on clean rows within the dataset
query = """
SELECT id, title, price, location, constituency, address, 
       bedrooms, bathrooms, floor_size, excerpt
FROM property_listings
WHERE price IS NOT NULL 
  AND floor_size IS NOT NULL
  AND price_flag IS NULL
  AND floor_size_flag IS NULL
"""

df = pd.read_sql(query, engine)
print(f"Clean rows for training: {len(df)}")
df.head()

Clean rows for training: 966


,id,title,price,location,constituency,address,bedrooms,bathrooms,floor_size,excerpt
0,1,4 Bedroom Townhouse,30000000.0,Kiambu Road,Westlands,"9 Kiambu Rd, Kiambu Road, Nairobi",4.0,5.0,250.0,4-Bedroom All En-Suite Townhouse for Sale on K...
1,2,4 Bedroom Apartment / Flat,65000000.0,Parklands,Westlands,"PRR9+25R Parklands, Parklands, Nairobi",4.0,5.0,250.0,4-Bedroom Duplex Apartment for Rent in Parklan...
2,3,4 Bedroom Apartment / Flat,65000000.0,Parklands,Westlands,"PRR9+25R Parklands, Parklands, Nairobi",4.0,5.0,250.0,LUXURY 4-BEDROOM DUPLEX APARTMENT FOR RENT & S...
3,4,5 Bedroom Apartment / Flat,40000000.0,Kileleshwa,Langata,"Olengruone Ave Kileleshwa, Kileleshwa, Nairobi",5.0,6.0,438.0,Newly Completed 5-Bedroom Duplex Penthouse for...
4,5,5 Bedroom Apartment / Flat,52000000.0,Kilimani,Langata,"PQ4H+8MC Elgeyo Rd, Kilimani, Nairobi",5.0,7.0,359.0,"FOR SALE\nSPACIOUS 3, 4, 5 BEDROOM SIMPLEX AP..."


In [7]:
# Check for potential duplicate addresses with different prices
dup_check = df.groupby('address').agg(
    count=('id', 'count'),
    price_min=('price', 'min'),
    price_max=('price', 'max')
).reset_index()

dup_check = dup_check[dup_check['count'] > 1].sort_values('count', ascending=False)
print(f"Addresses appearing more than once: {len(dup_check)}")
print(dup_check.head(20))

Addresses appearing more than once: 105
                                               address  count   price_min  \
203                      Westlands, Westlands, Nairobi     29   5000000.0   
34              Chiromo Ln Nairobi, Westlands, Nairobi     21   5000000.0   
184           Riverside Dr Nairobi, Riverside, Nairobi     19   7200000.0   
58             Kileleshwa Nairobi, Kileleshwa, Nairobi     17   6090000.0   
183      Ring Rd Parklands Nairobi, Westlands, Nairobi     10   6200000.0   
160         PRR4+F57 Spring Valley, Westlands, Nairobi     10   6200000.0   
47               JWRQ+85 Pridelands, Syokimau, Nairobi      9   4800000.0   
133           PQFP+XR5 Mandera Rd, Kileleshwa, Nairobi      9  29000000.0   
93      Olengruone Ave Kileleshwa, Kileleshwa, Nairobi      9   6661000.0   
62                         Kilimani, Kilimani, Nairobi      8   5600000.0   
81      Matundu Lane Spring Valley, Westlands, Nairobi      7   8700000.0   
180                 Riara Rd Maziwa,

In [8]:
# Check the exact rows that got removed
before = df_check_original = pd.read_csv('../data/property_listings_db.csv')
before_dedup = before.drop_duplicates(subset=['title', 'price', 'address', 'floor_size'], keep=False)
duplicated_rows = before[before.duplicated(subset=['title', 'price', 'address', 'floor_size'], keep=False)]

print(f"Rows involved in duplicates: {len(duplicated_rows)}")
print(duplicated_rows.sort_values('address').head(20)[['title', 'price', 'address', 'floor_size', 'bedrooms']])

Rows involved in duplicates: 599
                          title       price  \
616  3 Bedroom Apartment / Flat  17000000.0   
627  3 Bedroom Apartment / Flat  17000000.0   
729  1 Bedroom Apartment / Flat  10000000.0   
661  1 Bedroom Apartment / Flat  10000000.0   
12   3 Bedroom Apartment / Flat  49000000.0   
76   3 Bedroom Apartment / Flat  49000000.0   
116  2 Bedroom Apartment / Flat  14000000.0   
119  2 Bedroom Apartment / Flat  14000000.0   
948  1 Bedroom Apartment / Flat   6200000.0   
947  1 Bedroom Apartment / Flat   6200000.0   
611  1 Bedroom Apartment / Flat   6200000.0   
946  2 Bedroom Apartment / Flat  12000000.0   
945  2 Bedroom Apartment / Flat  12000000.0   
610  2 Bedroom Apartment / Flat  12000000.0   
189  1 Bedroom Apartment / Flat   7000000.0   
294  2 Bedroom Apartment / Flat  11500000.0   
241  2 Bedroom Apartment / Flat  11500000.0   
178  1 Bedroom Apartment / Flat   7000000.0   
312  4 Bedroom Apartment / Flat  31000000.0   
250  4 Bedroom Apartment / 

In [12]:
# Check if floor_size fix is even needed now
print(f"Rows with floor_size > 2000: {(train_df['floor_size'] > 2000).sum()}")

# Check price distribution without removing yet
print(f"\nRows above 95th percentile: {(train_df['price'] > train_df['price'].quantile(0.95)).sum()}")
print(f"Rows below 5th percentile: {(train_df['price'] < train_df['price'].quantile(0.05)).sum()}")

Rows with floor_size > 2000: 0

Rows above 95th percentile: 28
Rows below 5th percentile: 29
